In [1]:

import os
import requests
import pandas as pd
import logging
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now

# === LOGGING ===
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# === CONFIG ===

# 📌 Output files
output_dir = r"C:\Android Mobile App\Step 1_URL_Search"
output_csv_raw = os.path.join(output_dir, "github_android_search_results_raw.csv")
output_csv_filtered = os.path.join(output_dir, "github_android_search_results_filtered.csv")
output_ranges_csv = os.path.join(output_dir, "final_ranges_used.csv")

os.makedirs(output_dir, exist_ok=True)

# 📌 Auth
load_dotenv("All_Tokens.env")

# Dynamically read all tokens starting with 'GITHUB_TOKEN_' from env
tokens = [v for k, v in os.environ.items() if k.startswith("GITHUB_TOKEN_") and v]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0
HEADERS = {
    "Authorization": f"token {tokens[token_index]}",
    "Accept": "application/vnd.github+json"
}

# 📌 Date range
start_date = datetime.strptime("2025-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2025-01-31", "%Y-%m-%d")

# 📌 Window sizing (in hours!)
initial_window_hours = 30 * 24  # 30 days
min_window_hours = 1            # 1 hour minimum
max_window_hours = 90 * 24      # 90 days
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.2  # ~800/1000

# === Functions ===

def rotate_token():
    global token_index, HEADERS
    token_index = (token_index + 1) % len(tokens)
    HEADERS = {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github+json"
    }
    logger.info(f"🔑 Switched to token #{token_index + 1}")

def check_rate_limit():
    """Check Search API rate limit and sleep if needed."""
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if r.status_code != 200:
        logger.warning("⚠️  Could not check rate limit — proceeding cautiously.")
        return
    data = r.json()
    remaining = data['resources']['search']['remaining']
    reset_epoch = data['resources']['search']['reset']
    reset_in = max(0, reset_epoch - now())

    logger.info(f"🔎 Search API remaining: {remaining} requests | resets in {reset_in/60:.1f} min")

    if remaining < 5:
        logger.warning(f"⏳ Low quota. Sleeping for {reset_in/60:.1f} min until reset.")
        sleep(reset_in + 5)
        check_rate_limit()

def check_count(query):
    check_rate_limit()
    params = {"q": query, "per_page": 1}
    r = requests.get("https://api.github.com/search/repositories", headers=HEADERS, params=params)
    if r.status_code == 403:
        rotate_token()
        return check_count(query)
    if r.status_code != 200:
        logger.error(f"❌ Count check error: {r.status_code} — {r.text}")
        return -1
    return r.json().get("total_count", 0)

def fetch_items(query):
    all_items = []
    per_page = 100
    max_pages = 10

    for page in range(1, max_pages + 1):
        check_rate_limit()
        params = {"q": query, "per_page": per_page, "page": page}
        r = requests.get("https://api.github.com/search/repositories", headers=HEADERS, params=params)
        if r.status_code == 403:
            rotate_token()
            return fetch_items(query)
        if r.status_code != 200:
            logger.error(f"❌ Fetch error: {r.status_code} — {r.text}")
            break
        data = r.json().get("items", [])
        if not data:
            break
        all_items.extend(data)
        sleep(2)  # safe pacing
    return all_items

# === SMART LOOP ===

all_results = []
final_ranges = []

current_start = start_date
current_window_hours = initial_window_hours

# Define search variants: topic vs plain text
search_variants = [
    {"label": "topic", "extra": "topic:android"},
    {"label": "text", "extra": "android"}
]

while current_start < end_date:
    while True:
        current_end = current_start + timedelta(hours=current_window_hours)
        if current_end > end_date:
            current_end = end_date

        date_range = f"created:{current_start.isoformat()}..{current_end.isoformat()}"

        for variant in search_variants:
            base_query = (
                f"stars:>50 "
                f"(language:Kotlin OR language:Java OR language:Dart) "
                f"fork:false archived:false "
                f"{variant['extra']} "
                f"{date_range}"
            )

            total_count = check_count(base_query)
            logger.info(
                f"⏳ Checking [{variant['label']}] {current_start} to {current_end} "
                f"→ {total_count} repos (window {current_window_hours} hours)"
            )

            if total_count == -1:
                logger.warning(f"⚠️ Skipping [{variant['label']}] due to error.")
                continue

            items = fetch_items(base_query)
            items = [item for item in items if not item.get('fork', False) and not item.get('archived', False)]

            for item in items:
                item['search_qualifier'] = base_query
                item['repo_stars'] = item.get('stargazers_count', 0)
                item['match_type'] = variant['label']

            all_results.extend(items)
            final_ranges.append({
                "start_date": current_start.isoformat(),
                "end_date": current_end.isoformat(),
                "result_count": total_count,
                "window_hours": current_window_hours,
                "match_type": variant['label']
            })

            sleep(3)

        break

    current_start = current_end + timedelta(seconds=1)

# === SAVE ===

df = pd.json_normalize(all_results)
df.to_csv(output_csv_raw, index=False)
logger.info(f"✅ Raw repos saved to: {output_csv_raw}")

df_ranges = pd.DataFrame(final_ranges)
df_ranges.to_csv(output_ranges_csv, index=False)
logger.info(f"✅ Final ranges saved to: {output_ranges_csv}")

# === FILTER TO REQUIRED FIELDS ===

keep_fields = [
    'id', 'node_id', 'name', 'full_name', 'private', 'html_url', 'url',
    'clone_url', 'visibility', 'owner.login', 'size', 'stargazers_count',
    'watchers_count', 'language', 'forks', 'open_issues', 'default_branch',
    'open_issues_count', 'repo_stars', 'search_qualifier', 'match_type', 'topics', 'description'
]

for field in keep_fields:
    if field not in df.columns:
        df[field] = None

df = df[keep_fields]
df.to_csv(output_csv_filtered, index=False)
logger.info(f"✅ Filtered fields saved to: {output_csv_filtered}")


2025-06-21 21:15:24,712 [INFO] 🔎 Search API remaining: 30 requests | resets in 1.0 min
2025-06-21 21:15:24,907 [INFO] ⏳ Checking [topic] 2025-01-01 00:00:00 to 2025-01-31 00:00:00 → 0 repos (window 720 hours)
2025-06-21 21:15:25,030 [INFO] 🔎 Search API remaining: 29 requests | resets in 1.0 min
2025-06-21 21:15:28,364 [INFO] 🔎 Search API remaining: 28 requests | resets in 0.9 min
2025-06-21 21:15:28,570 [INFO] ⏳ Checking [text] 2025-01-01 00:00:00 to 2025-01-31 00:00:00 → 0 repos (window 720 hours)
2025-06-21 21:15:28,699 [INFO] 🔎 Search API remaining: 27 requests | resets in 0.9 min


KeyboardInterrupt: 